# Reconfigurable LLNN Inference on PYNQ-Z2

This notebook loads the **reconfigurable** LLNN overlay, programs all 4000 gates
with truth tables extracted from the trained model, then runs inference on
binarized 20×20 MNIST test data.

## Prerequisites
Upload these files to `/home/xilinx/reconfig/` on the PYNQ board:
- `llnn.bit` — bitstream (from `make build_reconfig`)
- `llnn.hwh` — hardware handoff (same base name!)
- `weights.json` — truth tables (from `scripts/extract_weights.py`)
- `binarized_20x20_MNIST.txt` — test data (10,000 × 400 binary digits)
- `mnist_test_labels.txt` — labels (one per line, 10,000 lines)

### AXI Address Map (axi_lut_ctrl.sv, 64KB)
| Offset | Description |
|--------|-------------|
| `0x0000–0x3FFC` | Gate programming (write `gate_id * 4` = 32-bit INIT) |
| `0x8000` | STATUS (R) — bit 0 = cfg_busy |
| `0x8004` | TOTAL_GATES (R) |
| `0x9000–0x9030` | NET_I input registers (13 × 32-bit words = 400 bits) |
| `0x9034` | NET_O output register (R) — lower 4 bits = class 0–9 |

## 1. Configuration

In [1]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BITSTREAM_PATH = "/home/xilinx/design_100.bit"
WEIGHTS_FILE   = "/home/xilinx/weights.json"
MNIST_DATA     = "/home/xilinx/jupyter_notebooks/pynq_notebooks/binarized_20x20_MNIST.txt"
LABELS_FILE    = "/home/xilinx/jupyter_notebooks/pynq_notebooks/mnist_test_labels.txt"

# ── AXI constants (must match axi_lut_ctrl.sv) ─────────────────────────────
ADDR_GATE_BASE   = 0x0000   # Gate programming: gate_id * 4
ADDR_STATUS      = 0x8000   # bit 0 = cfg_busy
ADDR_GATE_COUNT  = 0x8004   # total gates (R)
ADDR_INPUT_BASE  = 0x9000   # 13 × 32-bit input words start here
ADDR_OUTPUT      = 0x9034   # 4-bit classification output
NUM_INPUT_WORDS  = 13       # ceil(400 / 32)
NET_INPUTS       = 400      # 20×20 binarized pixels
AXI_ADDR_SPACE   = 0x10000  # 64 KB

## 2. Load the Overlay

In [2]:
from pynq import Overlay, MMIO

ol = Overlay(BITSTREAM_PATH)
print("Overlay loaded!")
print("IP dict keys:", list(ol.ip_dict.keys()))

Overlay loaded!
IP dict keys: ['llnn_wrapper_bd_0', 'processing_system7_0']


In [3]:
# Get MMIO handle to our LLNN AXI controller
ip_name = [k for k in ol.ip_dict.keys() if "llnn" in k.lower()]
if not ip_name:
    ip_name = [k for k in ol.ip_dict.keys() if "processing_system" not in k]
ip_name = ip_name[0]

base_addr = ol.ip_dict[ip_name]['phys_addr']
print(f"Using IP: {ip_name}")
print(f"Base address: 0x{base_addr:08X}")

mmio = MMIO(base_addr, AXI_ADDR_SPACE)

Using IP: llnn_wrapper_bd_0
Base address: 0x40000000


In [4]:
# Helper functions
def write_input(mmio, binary_string):
    """
    Write a 400-character binary string to the 13 AXI input registers.
    Bit 0 of the string maps to bit 0 of register 0 (LSB-first packing).
    """
    bits = [int(c) for c in binary_string[:NET_INPUTS]]
    for w in range(NUM_INPUT_WORDS):
        word = 0
        for b in range(32):
            idx = w * 32 + b
            if idx < len(bits):
                word |= bits[idx] << b
        mmio.write(ADDR_INPUT_BASE + w * 4, word)


def read_output(mmio):
    """Read the 4-bit classification result."""
    return mmio.read(ADDR_OUTPUT) & 0xF


def run_inference(mmio, binary_string):
    """Write input + read output in one call."""
    write_input(mmio, binary_string)
    return read_output(mmio)

## 3. Smoke Test — Read Status Registers

In [5]:
total_gates = mmio.read(ADDR_GATE_COUNT)
status = mmio.read(ADDR_STATUS)
print(f"TOTAL_GATES = {total_gates} (expected: 4000)")
print(f"STATUS      = 0x{status:08X} (expected: 0 = idle)")
assert total_gates == 4000, f"Expected 4000 gates, got {total_gates}"
print("✅ Smoke test passed!")

TOTAL_GATES = 4000 (expected: 4000)
STATUS      = 0x00000000 (expected: 0 = idle)
✅ Smoke test passed!


## 4. Load MNIST Data

In [6]:
import os

with open(MNIST_DATA, "r") as f:
    samples = [line.strip() for line in f.readlines()]

print(f"Loaded {len(samples)} samples")
print(f"Sample length: {len(samples[0])} bits")

labels = None
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE, "r") as f:
        labels = [int(line.strip()) for line in f.readlines()]
    print(f"Loaded {len(labels)} labels")
else:
    print("No labels file found.")

Loaded 10000 samples
Sample length: 400 bits
Loaded 10000 labels


## 5. Untrained Performance: 0xDEADBEEF in LUT INITs

In [7]:
import time
NUM_SAMPLES = min(len(samples), 1000)

predictions = []
t0 = time.time()

for i in range(NUM_SAMPLES):
    pred = run_inference(mmio, samples[i])
    predictions.append(pred)

elapsed = time.time() - t0
print(f"Ran {NUM_SAMPLES} inferences in {elapsed:.3f} s")
print(f"Throughput: {NUM_SAMPLES / elapsed:.1f} samples/sec")
print(f"Latency:    {elapsed / NUM_SAMPLES * 1000:.2f} ms/sample")

if labels:
    correct = sum(p == labels[i] for i, p in enumerate(predictions))
    accuracy = correct / NUM_SAMPLES * 100
    print(f"\nAccuracy: {correct}/{NUM_SAMPLES} = {accuracy:.2f}%")

Ran 1000 inferences in 3.018 s
Throughput: 331.4 samples/sec
Latency:    3.02 ms/sample

Accuracy: 93/1000 = 9.30%


## 6. Program All Gates from weights.json

In [8]:
import json

def wait_ready(mmio, timeout_ms=100):
    """Wait for configuration shift FSM to finish."""
    start = time.time()
    while mmio.read(ADDR_STATUS) & 0x1:
        if (time.time() - start) * 1000 > timeout_ms:
            raise TimeoutError("cfg_busy timeout!")
        time.sleep(0.0001)


def program_gate(mmio, gate_id, init_value):
    """Program a single gate's truth table (32-bit INIT)."""
    wait_ready(mmio)
    mmio.write(ADDR_GATE_BASE + gate_id * 4, init_value & 0xFFFFFFFF)
    wait_ready(mmio)


# Load weights
with open(WEIGHTS_FILE, "r") as f:
    weights = json.load(f)

gates = weights["gates"]
print(f"Loaded {len(gates)} gate truth tables from {WEIGHTS_FILE}")

# Program all gates
t0 = time.time()
for g in gates:
    program_gate(mmio, g["gate_id"], g["init"])
elapsed = time.time() - t0

print(f"✅ Programmed {len(gates)} gates in {elapsed:.2f}s ({len(gates)/elapsed:.0f} gates/sec)")

Loaded 4000 gate truth tables from /home/xilinx/weights.json
✅ Programmed 4000 gates in 0.37s (10701 gates/sec)


## 7. Quick Smoke Test — Write Input + Read Output

In [9]:
# Write all-zero input
write_input(mmio, "0" * NET_INPUTS)
result = read_output(mmio)
print(f"All-zeros input → class {result}")

# Write all-ones input
write_input(mmio, "1" * NET_INPUTS)
result = read_output(mmio)
print(f"All-ones input  → class {result}")

print("\n✅ AXI communication working!" if result <= 9 else "\n❌ Unexpected output")

All-zeros input → class 1
All-ones input  → class 8

✅ AXI communication working!


## 8. Visualize a Sample

In [10]:
def show_sample(binary_string, idx=None, label=None, prediction=None):
    """Display a 20×20 binarized image as text art."""
    title = f"Sample {idx}" if idx is not None else "Sample"
    if label is not None:
        title += f" | Label: {label}"
    if prediction is not None:
        match = "✅" if prediction == label else "❌"
        title += f" | Predicted: {prediction} {match}"
    print(title)
    print("─" * 22)
    for row in range(20):
        line = binary_string[row * 20 : (row + 1) * 20]
        print(" " + line.replace("0", "· ").replace("1", "  "))
    print()

# Show first sample
pred = run_inference(mmio, samples[0])
show_sample(samples[0], idx=0, label=labels[0] if labels else None, prediction=pred)

Sample 0 | Label: 7 | Predicted: 7 ✅
──────────────────────
 · ·             · · · · · · · · · · · · 
 · ·                                 · · 
 · ·                                 · · 
 · · · · · · ·                       · · 
 · · · · · · · · · · · · · ·         · · 
 · · · · · · · · · · · · ·         · · · 
 · · · · · · · · · · · · ·         · · · 
 · · · · · · · · · · · ·         · · · · 
 · · · · · · · · · · · ·         · · · · 
 · · · · · · · · · · ·         · · · · · 
 · · · · · · · · · · ·       · · · · · · 
 · · · · · · · · · ·         · · · · · · 
 · · · · · · · · ·         · · · · · · · 
 · · · · · · · ·           · · · · · · · 
 · · · · · · · ·         · · · · · · · · 
 · · · · · · ·           · · · · · · · · 
 · · · · · · ·         · · · · · · · · · 
 · · · · · ·           · · · · · · · · · 
 · · · · · ·           · · · · · · · · · 
 · · · · · ·         · · · · · · · · · · 



## 9. Run Batch Inference with Trained LUTs

In [11]:
NUM_SAMPLES = min(len(samples), 1000)

predictions = []
t0 = time.time()

for i in range(NUM_SAMPLES):
    pred = run_inference(mmio, samples[i])
    predictions.append(pred)

elapsed = time.time() - t0
print(f"Ran {NUM_SAMPLES} inferences in {elapsed:.3f} s")
print(f"Throughput: {NUM_SAMPLES / elapsed:.1f} samples/sec")
print(f"Latency:    {elapsed / NUM_SAMPLES * 1000:.2f} ms/sample")

if labels:
    correct = sum(p == labels[i] for i, p in enumerate(predictions))
    accuracy = correct / NUM_SAMPLES * 100
    print(f"\nAccuracy: {correct}/{NUM_SAMPLES} = {accuracy:.2f}%")

Ran 1000 inferences in 2.693 s
Throughput: 371.3 samples/sec
Latency:    2.69 ms/sample

Accuracy: 922/1000 = 92.20%


## 10. Confusion Matrix

In [12]:
if labels:
    matrix = [[0] * 10 for _ in range(10)]
    for i in range(NUM_SAMPLES):
        true_label = labels[i]
        pred_label = predictions[i]
        if true_label < 10 and pred_label < 10:
            matrix[true_label][pred_label] += 1

    print("Confusion Matrix (rows = true, cols = predicted)")
    print("     " + "  ".join(f"{j:3d}" for j in range(10)))
    print("    " + "─" * 45)
    for i in range(10):
        row = "  ".join(f"{matrix[i][j]:3d}" for j in range(10))
        print(f" {i}  │{row}")
else:
    print("Skipping — no labels.")

Confusion Matrix (rows = true, cols = predicted)
       0    1    2    3    4    5    6    7    8    9
    ─────────────────────────────────────────────
 0  │ 83    0    0    0    0    1    1    0    0    0
 1  │  0  120    0    0    0    0    2    0    4    0
 2  │  0    0  109    1    0    0    0    3    3    0
 3  │  0    0    3   99    0    2    0    1    1    1
 4  │  1    0    1    0   96    0    2    0    2    8
 5  │  1    0    0    2    1   78    1    0    4    0
 6  │  3    0    0    0    0    2   81    0    1    0
 7  │  0    0    2    1    1    0    0   87    2    6
 8  │  0    0    1    2    0    0    0    0   85    1
 9  │  0    0    0    0    0    0    0    2    8   84


## 11. Interactive — Classify Specific Samples

In [13]:
for idx in [0, 1, 2, 3, 4, 100, 500, 999]:
    pred = run_inference(mmio, samples[idx])
    label = labels[idx] if labels else None
    show_sample(samples[idx], idx=idx, label=label, prediction=pred)

Sample 0 | Label: 7 | Predicted: 7 ✅
──────────────────────
 · ·             · · · · · · · · · · · · 
 · ·                                 · · 
 · ·                                 · · 
 · · · · · · ·                       · · 
 · · · · · · · · · · · · · ·         · · 
 · · · · · · · · · · · · ·         · · · 
 · · · · · · · · · · · · ·         · · · 
 · · · · · · · · · · · ·         · · · · 
 · · · · · · · · · · · ·         · · · · 
 · · · · · · · · · · ·         · · · · · 
 · · · · · · · · · · ·       · · · · · · 
 · · · · · · · · · ·         · · · · · · 
 · · · · · · · · ·         · · · · · · · 
 · · · · · · · ·           · · · · · · · 
 · · · · · · · ·         · · · · · · · · 
 · · · · · · ·           · · · · · · · · 
 · · · · · · ·         · · · · · · · · · 
 · · · · · ·           · · · · · · · · · 
 · · · · · ·           · · · · · · · · · 
 · · · · · ·         · · · · · · · · · · 

Sample 1 | Label: 2 | Predicted: 2 ✅
──────────────────────
 · · ·               · · · · · · · · · 

## 12. Runtime Reconfiguration Demo

The power of the reconfigurable overlay: reprogram gates **without re-synthesizing**.
Below we demonstrate reprogramming gate 0 and verifying the output changes.

In [ ]:
# Save original output for sample 0
write_input(mmio, samples[0])
original_output = read_output(mmio)
print(f"Original prediction for sample 0: class {original_output}")

# Corrupt gate 0 to all-zeros truth table
program_gate(mmio, 0, 0x00000000)
write_input(mmio, samples[0])
corrupted_output = read_output(mmio)
print(f"After zeroing gate 0: class {corrupted_output}")

# Restore gate 0
original_init = gates[0]["init"]
program_gate(mmio, 0, original_init)
write_input(mmio, samples[0])
restored_output = read_output(mmio)
print(f"After restoring gate 0: class {restored_output}")

assert restored_output == original_output, "Gate restore failed!"
print("\n✅ Runtime reconfiguration verified!")

## 13. Hot-Swap to Rotated MNIST Model

We now load the `weights_rot.json` truth tables and reprogram the entire network over AXI. No Vivado needed!

In [14]:
WEIGHTS_ROT_FILE = "/home/xilinx/weights_rot.json"

with open(WEIGHTS_ROT_FILE, "r") as f:
    weights_rot = json.load(f)

gates_rot = weights_rot["gates"]
print(f"Loaded {len(gates_rot)} gate truth tables from {WEIGHTS_ROT_FILE}")

# Hot-swap all gates
t0 = time.time()
for g in gates_rot:
    program_gate(mmio, g["gate_id"], g["init"])
elapsed = time.time() - t0

print(f"✅ Hot-swapped {len(gates_rot)} gates in {elapsed:.2f}s ({len(gates_rot)/elapsed:.0f} gates/sec)")

Loaded 4000 gate truth tables from /home/xilinx/weights_rot.json
✅ Hot-swapped 4000 gates in 0.37s (10685 gates/sec)


## 14. Evaluate on Rotated Dataset

Now we load the specially rotated 90-degree MNIST test images and evaluate the newly programmed network.

In [15]:
MNIST_ROT_DATA = "/home/xilinx/jupyter_notebooks/pynq_notebooks/binarized_20x20_MNIST_rotated.txt"
LABELS_ROT_FILE = "/home/xilinx/jupyter_notebooks/pynq_notebooks/mnist_rotated_test_labels.txt"

with open(MNIST_ROT_DATA, "r") as f:
    rot_samples = [line.strip() for line in f.readlines()]

with open(LABELS_ROT_FILE, "r") as f:
    rot_labels = [int(line.strip()) for line in f.readlines()]

NUM_SAMPLES = min(len(rot_samples), 1000)
predictions_rot = []
t0 = time.time()

for i in range(NUM_SAMPLES):
    pred = run_inference(mmio, rot_samples[i])
    predictions_rot.append(pred)

elapsed = time.time() - t0
print(f"Ran {NUM_SAMPLES} inferences in {elapsed:.3f} s")

correct = sum(p == rot_labels[i] for i, p in enumerate(predictions_rot))
accuracy = correct / NUM_SAMPLES * 100
print(f"\nAccuracy on Rotated MNIST: {correct}/{NUM_SAMPLES} = {accuracy:.2f}%")

Ran 1000 inferences in 2.733 s

Accuracy on Rotated MNIST: 937/1000 = 93.70%


## 15. Interactive — Classify Specific Rotated Samples

Visualize specific rotated MNIST images and check their individual predictions.

In [16]:
for idx in [0, 1, 2, 3, 4, 100, 500, 999]:
    pred = run_inference(mmio, rot_samples[idx])
    label = rot_labels[idx] if rot_labels else None
    show_sample(rot_samples[idx], idx=idx, label=label, prediction=pred)


Sample 0 | Label: 7 | Predicted: 7 ✅
──────────────────────
 · · · · · · · · · · · · · · · · · · · · 
 · · · · · · · · · · · · · · · · · · · · 
 ·         · · · · · · · · · · · · · · · 
 ·             · · · · · · · · · · · · · 
 ·                 · · · · · · · · · · · 
 ·                   · · · · · · · · · · 
 ·       ·               · · · · · · · · 
 ·       · · ·               · · · · · · 
 ·       · · · · ·               · · · · 
 ·       · · · · · · ·                 · 
 ·       · · · · · · · ·                 
 ·       · · · · · · · · ·               
         · · · · · · · · · · ·           
       · · · · · · · · · · · · · ·       
       · · · · · · · · · · · · · · · · · 
       · · · · · · · · · · · · · · · · · 
       · · · · · · · · · · · · · · · · · 
       · · · · · · · · · · · · · · · · · 
 · · · · · · · · · · · · · · · · · · · · 
 · · · · · · · · · · · · · · · · · · · · 

Sample 1 | Label: 2 | Predicted: 2 ✅
──────────────────────
 · · · · · · · · · · · · · · · ·       